# RGCA Baseline Experiment Notebook

This notebook runs the structured pre-RGCA baseline experiments for retrieval-induced hallucination in chest X-ray report generation.

Baseline pipeline:

```text
Image/study record -> Retriever -> Retrieved reports -> Generator -> Generated report -> Evaluation
```

The purpose is not to beat SOTA yet. The purpose is to create a reproducible experiment setup that shows retrieval behavior, mismatch behavior, and the evaluation artifacts needed before implementing RGCA.


## What This Notebook Produces

If run end-to-end, this notebook produces:

- a MIMIC pilot subset JSONL, if raw MIMIC files are available
- a structured experiment suite with named runs
- retrieval outputs for clean and mismatch retrieval
- generated reports for no-retrieval, retrieval, and mismatch modes
- hallucination evaluation summaries
- CSV/Markdown result tables
- a private zip artifact for download

Important: MIMIC-CXR data and derived report text must remain private. Do not publish raw reports, images, or generated outputs containing report text to GitHub.


## 1. Environment Setup

Run this cell first. It detects whether we are in Kaggle or a local checkout and sets paths accordingly.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

IS_KAGGLE = Path('/kaggle/working').exists()

USE_DEMO_DATA = not IS_KAGGLE

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/RGCA')
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROJECT_ROOT exists:', PROJECT_ROOT.exists())


## 2. Clone Or Update The Repository On Kaggle

Run this only on Kaggle. If you are local, skip it.


In [ ]:
if IS_KAGGLE:
    if not PROJECT_ROOT.exists():
        subprocess.run(['git', 'clone', 'https://github.com/pidoxy/RGCA.git', str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
else:
    print('Local environment detected. Skipping clone/pull/install cell.')


## 3. Import RGCA Helpers


In [ ]:
from rgca_baseline.io_utils import read_jsonl
from rgca_baseline.pipeline import load_studies

def run_command(command, cwd=PROJECT_ROOT):
    print('+', ' '.join(str(part) for part in command))
    return subprocess.run(command, cwd=str(cwd), check=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def print_json(payload):
    print(json.dumps(payload, indent=2))

print('Imports ready')


## 4. Configure Data Paths

These defaults match the paths we have been using in Kaggle. Adjust them if your uploaded dataset is attached under `/kaggle/input/...` instead of `/kaggle/working/physionet/...`.


In [ ]:
if IS_KAGGLE:
    PHYSIONET_ROOT = Path('/kaggle/working/physionet')
    MIMIC_REPORTS_ROOT = PHYSIONET_ROOT / 'mimic-cxr' / 'files'
    MIMIC_JPG_ROOT = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'files'
    METADATA_PATH = PHYSIONET_ROOT / 'mimic-cxr-2.0.0-metadata.csv.gz'
    SPLIT_PATH = PHYSIONET_ROOT / 'mimic-cxr-2.0.0-split.csv.gz'
    LABELS_PATH = PHYSIONET_ROOT / 'mimic-cxr-2.0.0-chexpert.csv.gz'
    PILOT_OUTPUT_DIR = Path('/kaggle/working/rgca_pilot_500')
    SUITE_OUTPUT_DIR = Path('/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0')
else:
    PHYSIONET_ROOT = PROJECT_ROOT / 'data' / 'raw'
    MIMIC_REPORTS_ROOT = PHYSIONET_ROOT / 'mimic-cxr' / 'files'
    MIMIC_JPG_ROOT = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'files'
    METADATA_PATH = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'mimic-cxr-2.0.0-metadata.csv.gz'
    SPLIT_PATH = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'mimic-cxr-2.0.0-split.csv.gz'
    LABELS_PATH = PHYSIONET_ROOT / 'mimic-cxr-jpg' / 'mimic-cxr-2.0.0-chexpert.csv.gz'
    PILOT_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'rgca_pilot_500'
    SUITE_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'rgca_experiments' / 'notebook_demo_suite'

SUBSET_PATH = PROJECT_ROOT / 'data' / 'demo' / 'demo_studies.jsonl' if USE_DEMO_DATA else PILOT_OUTPUT_DIR / 'data' / 'mimic_subset.jsonl'
SUITE_CONFIG = PROJECT_ROOT / 'configs' / 'mimic_pilot_suite.json'

paths = {
    'METADATA_PATH': METADATA_PATH,
    'SPLIT_PATH': SPLIT_PATH,
    'LABELS_PATH': LABELS_PATH,
    'MIMIC_REPORTS_ROOT': MIMIC_REPORTS_ROOT,
    'MIMIC_JPG_ROOT': MIMIC_JPG_ROOT,
    'SUBSET_PATH': SUBSET_PATH,
    'SUITE_CONFIG': SUITE_CONFIG,
    'SUITE_OUTPUT_DIR': SUITE_OUTPUT_DIR,
}

print('USE_DEMO_DATA:', USE_DEMO_DATA)
for name, path in paths.items():
    print(f'{name}: {path} | exists={Path(path).exists()}')


## 5. Choose Pilot Size

Use 500 studies for the current working pilot. This creates 400 retrieval-pool studies and 100 evaluation studies.

For a quick smoke test, change these to `RETRIEVAL_LIMIT = 80` and `EVAL_LIMIT = 20`.


In [ ]:
RETRIEVAL_LIMIT = 400
EVAL_LIMIT = 100
TOP_K = 3

print('retrieval_limit:', RETRIEVAL_LIMIT)
print('eval_limit:', EVAL_LIMIT)
print('top_k:', TOP_K)


## 6. Build The Pilot Subset

Run this if `SUBSET_PATH` does not already exist. The output is a study-level JSONL with image/report pairing and train/validate split mapping.


In [ ]:
if USE_DEMO_DATA:
    print('Using demo dataset for local smoke testing:', SUBSET_PATH)
elif SUBSET_PATH.exists():
    print('Subset already exists:', SUBSET_PATH)
else:
    command = [
        sys.executable,
        'scripts/kaggle_run_pilot.py',
        '--metadata', str(METADATA_PATH),
        '--split', str(SPLIT_PATH),
        '--labels', str(LABELS_PATH),
        '--reports-root', str(MIMIC_REPORTS_ROOT),
        '--images-root', str(MIMIC_JPG_ROOT),
        '--output-dir', str(PILOT_OUTPUT_DIR),
        '--limit', str(RETRIEVAL_LIMIT + EVAL_LIMIT),
        '--retrieval-limit', str(RETRIEVAL_LIMIT),
        '--eval-limit', str(EVAL_LIMIT),
        '--retriever', 'lexical',
        '--generator', 'mock',
        '--top-k', str(TOP_K),
    ]
    run_command(command)

print('Subset path:', SUBSET_PATH)
print('Subset exists:', SUBSET_PATH.exists())


## 7. Validate The Subset

This confirms that we have both retrieval-pool and evaluation records. If `eval` is zero, the experiment is invalid.


In [ ]:
studies = load_studies(SUBSET_PATH)
retrieval_pool = [study for study in studies if study.split == 'retrieval_pool']
eval_studies = [study for study in studies if study.split == 'eval']

print('total_studies:', len(studies))
print('retrieval_pool:', len(retrieval_pool))
print('eval_studies:', len(eval_studies))
print('first_record:', studies[0].to_dict() if studies else None)

assert retrieval_pool, 'Invalid subset: no retrieval_pool records found.'
assert eval_studies, 'Invalid subset: no eval records found.'


## 8. Run The Structured Experiment Suite

This is now the preferred path. It runs the planned experiment matrix from `configs/mimic_pilot_suite.json` and writes one output folder per experiment.


In [ ]:
run_command([
    sys.executable,
    'scripts/run_experiment_suite.py',
    '--config', str(SUITE_CONFIG),
    '--input', str(SUBSET_PATH),
    '--output-dir', str(SUITE_OUTPUT_DIR),
    '--overwrite',
])


## 9. Summarize Results Into Tables


In [ ]:
TABLES_DIR = SUITE_OUTPUT_DIR / 'tables'
SUITE_MANIFEST = SUITE_OUTPUT_DIR / 'suite_manifest.json'

run_command([
    sys.executable,
    'scripts/summarize_experiment_suite.py',
    '--manifest', str(SUITE_MANIFEST),
    '--output-dir', str(TABLES_DIR),
])

print('Markdown table:', TABLES_DIR / 'suite_summary.md')
print('CSV table:', TABLES_DIR / 'suite_summary.csv')


## 10. Display The Summary Table


In [ ]:
summary_md = (TABLES_DIR / 'suite_summary.md').read_text(encoding='utf-8')
print(summary_md[:6000])


## 11. Inspect The Suite Manifest

The manifest is the reproducibility record. It stores the subset path, output path, and settings/results for every named experiment.


In [ ]:
suite_manifest = read_json(SUITE_MANIFEST)
print('suite_name:', suite_manifest['suite_name'])
print('input_path:', suite_manifest['input_path'])
print('num_experiments:', len(suite_manifest['experiments']))

for experiment in suite_manifest['experiments']:
    pipeline = experiment['pipeline_summary']
    print({
        'name': experiment['name'],
        'retriever': pipeline['retriever_backend'],
        'generator': pipeline['generator_backend'],
        'top_k': pipeline['top_k'],
        'generated_counts': pipeline['generated_counts'],
    })


## 12. Inspect Mismatch Examples

Use this to manually review examples where retrieved labels appear in generated labels. Start with the controlled stress experiment, then later repeat for a real VLM experiment.


In [ ]:
EXPERIMENT_TO_INSPECT = 'E02_stress_lexical_k3'
DETAILS_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'evaluation' / 'mismatch' / 'evaluation_details.jsonl'
GENERATIONS_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'baseline' / 'generations_mismatch.jsonl'
RETRIEVAL_PATH = SUITE_OUTPUT_DIR / EXPERIMENT_TO_INSPECT / 'baseline' / 'mismatch_results.jsonl'

details = read_jsonl(DETAILS_PATH)
generations = {row['study_id']: row for row in read_jsonl(GENERATIONS_PATH)}
retrievals = {row['target_study']: row for row in read_jsonl(RETRIEVAL_PATH)}

interesting = [row for row in details if row['retrieval_induced_flags']]
print('cases_with_retrieval_induced_flags:', len(interesting))

for row in interesting[:5]:
    study_id = row['study_id']
    print('\n' + '=' * 100)
    print('study_id:', study_id)
    print('reference_labels:', row['reference_labels'])
    print('retrieved_labels:', row['retrieved_labels'])
    print('generated_labels:', row['generated_labels'])
    print('retrieval_induced_flags:', row['retrieval_induced_flags'])
    print('\nGenerated report snippet:')
    print(generations[study_id]['generated_report'][:1500])
    print('\nFirst retrieved report snippet:')
    print(retrievals[study_id]['retrieved_reports'][0][:1500])


## 13. Compare `k = 1, 3, 5`

This lets us quickly check whether increasing retrieved context increases copied unsupported findings in the controlled stress setup.


In [ ]:
for experiment in suite_manifest['experiments']:
    name = experiment['name']
    if not name.startswith('E0') or 'stress_lexical' not in name:
        continue
    pipeline = experiment['pipeline_summary']
    mismatch = experiment['evaluation_summaries']['mismatch']['mismatch']
    print({
        'experiment': name,
        'top_k': pipeline['top_k'],
        'retrieval_induced_hallucination_rate': mismatch['retrieval_induced_hallucination_rate'],
        'retrieval_copy_rate': mismatch['retrieval_copy_rate'],
        'total_retrieval_induced_hallucinations': mismatch['total_retrieval_induced_hallucinations'],
    })


## 14. Package Private Artifacts For Download

This zip can be downloaded from Kaggle output. Keep it private because it may contain MIMIC report text.


In [ ]:
ZIP_PATH = SUITE_OUTPUT_DIR.parent / f'{SUITE_OUTPUT_DIR.name}.zip'

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', root_dir=str(SUITE_OUTPUT_DIR))
print('Created:', ZIP_PATH)
print('Size MB:', round(ZIP_PATH.stat().st_size / (1024 * 1024), 2))


## 15. Optional: Run A Single Experiment

Use this when debugging one run without rerunning the entire suite.


In [ ]:
# Example only. Uncomment to run a single experiment.
# run_command([
#     sys.executable,
#     'scripts/run_experiment_suite.py',
#     '--config', str(SUITE_CONFIG),
#     '--input', str(SUBSET_PATH),
#     '--output-dir', str(SUITE_OUTPUT_DIR),
#     '--only', 'E02_stress_lexical_k3',
#     '--overwrite',
# ])


## 16. How To Interpret Current Results

Use this language carefully:

- `mock` generator runs validate infrastructure only.
- `retrieval_copy_stress` runs validate the mismatch/evaluation protocol under controlled contamination.
- These are not final clinical results.
- The paper-level empirical claim requires real image/text retrieval and a real VLM generator.

Current evidence tier:

```text
Tier 1: infrastructure evidence      -> complete
Tier 2: controlled failure evidence  -> complete after stress suite
Tier 3: real VLM evidence            -> next
```


## 17. Next Implementation Steps

After this notebook runs successfully:

1. Save the private zip artifact.
2. Use `suite_summary.csv` for early tables.
3. Manually review 10 to 20 mismatch cases.
4. Implement real image/text retrieval, ideally BioMedCLIP-style retrieval.
5. Add a real VLM backend and rerun the same structured suite.

This keeps us disciplined: same subset, same matrix, same output contract, stronger backends over time.
